# 13 — DeepSeekMoE

**Paper:** DeepSeekMoE + used in V2 FFN.

GPT-2 / llm.c: **one** MLP per block (`matmul` + GELU + `matmul`).

MoE: many **expert** MLPs; a **router** picks top-k experts per token. **Shared** experts always run.


In [ ]:
import torch
from llmc.deepseek_v2 import DeepSeekV2Config, DeepSeekMoE

cfg = DeepSeekV2Config.tiny(vocab_size=128, block_size=32)
x = torch.randn(2, 16, cfg.n_embd)
moe = DeepSeekMoE(cfg)
y = moe(x)

print("MoE in:", tuple(x.shape), "out:", tuple(y.shape))
print("routed experts:", cfg.n_routed_experts, "top-k:", cfg.num_experts_per_tok, "shared:", cfg.n_shared_experts)


In [ ]:
# Peek at routing weights for one token
flat = x.view(-1, cfg.n_embd)
logits = moe.gate(flat[0:1])
probs = torch.softmax(logits, dim=-1)
topw, topi = torch.topk(probs, cfg.num_experts_per_tok)
print("token 0 expert indices:", topi.tolist())
print("token 0 expert weights:", topw.round(decimals=3).tolist())


## C port (next piece)

After MLA (`c/deepseek_v2/mla.c`), we will add **`moe.c`** with the same comment style as `DeepSeekMoE` in Python.

GPT-2 in llm.c has **no router** — this block is entirely new compared to `train_gpt2.c`.